# Week 23, Day 3: Coupled oscillators and synchronisation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w023/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Lock or drift

Simulate the two-oscillator model with ω₁ = 0 and ω₂ = 1 (Euler, dt = 0.001, 100 time units) for K = 2 and K = 0.8. Store the final phase difference θ₂ − θ₁ (wrapped to between −π and π) in `phi_end[K]`, and for K = 0.8 the average rate at which the unwrapped difference grows over the second half in `drift_rate`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
w1, w2, dt, steps = 0.0, 1.0, 0.001, 100000
phi_end, traces = {}, {}
for K in (2.0, 0.8):
    t1, t2 = 0.0, 0.0; ph = np.empty(steps)
    for k in range(steps):
        d1 = w1 + K / 2 * np.sin(t2 - t1)
        d2 = ...
        t1 += dt * d1; t2 += dt * d2; ph[k] = t2 - t1
    traces[K] = ph
    phi_end[K] = np.angle(np.exp(1j * ph[-1]))
half = steps // 2
drift_rate = ...
print(f"K = 2: locked at {phi_end[2.0]:.4f} (arcsin 0.5 = {np.arcsin(0.5):.4f});  K = 0.8: drifts at {drift_rate:.4f}")
for K in traces: plt.plot(np.arange(steps) * dt, traces[K], label=f"K = {K}")
plt.xlabel("time"); plt.ylabel("theta2 - theta1"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_lock():
    "with K = 2 the phase difference settles within 1e-3 of arcsin(1/2)"
    assert abs(phi_end[2.0] - np.arcsin(0.5)) < 1e-3

def test_drift():
    "with K = 0.8 the phase difference drifts at the beat frequency sqrt(1 - 0.64) = 0.6, within 2%"
    assert abs(drift_rate - 0.6) / 0.6 < 0.02

test_lock()
test_drift()
print("All checks passed")

### Exercise: The onset of synchrony

Give 1,000 oscillators Lorentzian frequencies of width γ = 0.5 (so Kc = 1), placed at evenly spaced quantiles, ω = γ tan(π(q − ½)). For K = 0.5, 1.5, 2 and 3, start from random phases, simulate with Euler steps of 0.05 for 6,000 steps using the order-parameter form, and store the mean R over the last 3,000 steps in `R_mean[K]`.

In [ ]:
N, gam = 1000, 0.5
q = (np.arange(N) + 0.5) / N
om = gam * np.tan(np.pi * (q - 0.5))
R_mean, final_th = {}, {}
for K in (0.5, 1.5, 2.0, 3.0):
    r_k = np.random.default_rng(1975)
    th = r_k.uniform(0, 2 * np.pi, N); Rs = []
    for k in range(6000):
        z = np.mean(np.exp(1j * th)); R, psi = abs(z), np.angle(z)
        th = ...
        if k >= 3000: Rs.append(R)
    R_mean[K] = np.mean(Rs); final_th[K] = th
print({K: round(float(v), 3) for K, v in R_mean.items()}, " simulated;", {K: round(float(np.sqrt(max(0, 1 - 1 / K))), 3) for K in R_mean}, " theory")

In [ ]:
import numpy as np
def test_below():
    "below the critical coupling (K = 0.5) the order parameter stays below 0.1"
    assert R_mean[0.5] < 0.1

def test_above():
    "above it, R is within 0.02 of sqrt(1 - Kc / K) for K = 1.5, 2 and 3"
    for K in (1.5, 2.0, 3.0):
        assert abs(R_mean[K] - np.sqrt(1 - 1.0 / K)) < 0.02

test_below()
test_above()
print("All checks passed")

### Exercise: Locked and drifting

For K = 2, run the simulation again from the same start for 6,000 steps, storing the phases at step 3,000. The effective frequency of each oscillator is its phase change over the last 3,000 steps divided by the time. Store the fraction with |effective frequency| below 0.02 in `frac_locked`, and the fraction with |ω| ≤ K·R_mean[2] in `frac_pred`.

In [ ]:
K = 2.0
r_k = np.random.default_rng(1975)
th = r_k.uniform(0, 2 * np.pi, N)
for k in range(6000):
    z = np.mean(np.exp(1j * th)); R, psi = abs(z), np.angle(z)
    th = th + 0.05 * (om + K * R * np.sin(psi - th))
    if k == 2999: th_mid = th.copy()
eff = ...                                        # effective frequency of each oscillator
frac_locked = ...
frac_pred = ...
print(f"locked fraction {frac_locked:.3f}, predicted {frac_pred:.3f}")
plt.plot(om, eff, ".", ms=2); plt.xlim(-5, 5); plt.xlabel("natural frequency"); plt.ylabel("effective frequency"); plt.show()

In [ ]:
import numpy as np
def test_fraction():
    "the fraction of oscillators with effective frequency below 0.02 matches the fraction with |omega| at most K R, within 0.03"
    assert abs(frac_locked - frac_pred) < 0.03

test_fraction()
print("All checks passed")